# MovieLens 20M — Data Exploration (Stage 1)

Explore `ratings.csv`, `movies.csv`, and `tags.csv` before locking the Elasticsearch schema.

**Raw data path:** `../data/raw/`

**Kernel:** use project venv — `Python (bigData)` or `.venv/Scripts/python.exe`

Run all cells top-to-bottom. Full dataset (~20M ratings) may take a few minutes to load.

In [5]:
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_RAW = Path("../data/raw")
RATINGS_PATH = DATA_RAW / "ratings.csv"
MOVIES_PATH = DATA_RAW / "movies.csv"
TAGS_PATH = DATA_RAW / "tags.csv"

for path in (RATINGS_PATH, MOVIES_PATH, TAGS_PATH):
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

print("Data files found.")

ModuleNotFoundError: No module named 'pandas'

In [ ]:
ratings = pd.read_csv(RATINGS_PATH)
movies = pd.read_csv(MOVIES_PATH)
tags = pd.read_csv(TAGS_PATH)

print(f"ratings: {len(ratings):,} rows")
print(f"movies:  {len(movies):,} rows")
print(f"tags:    {len(tags):,} rows")

## 1. Ratings analysis (`ratings.csv`)

Columns: `userId`, `movieId`, `rating`, `timestamp`

In [ ]:
ratings.head()

In [ ]:
ratings.info()

In [ ]:
print("Missing values:")
print(ratings.isna().sum())

duplicate_rows = ratings.duplicated().sum()
print(f"\nDuplicate rows: {duplicate_rows:,}")

print(f"\nUnique users:  {ratings['userId'].nunique():,}")
print(f"Unique movies: {ratings['movieId'].nunique():,}")

In [ ]:
print("Rating distribution:")
print(ratings["rating"].describe())
print("\nValue counts:")
print(ratings["rating"].value_counts().sort_index())

valid_half_steps = ratings["rating"].mod(0.5).eq(0).all()
in_range = ratings["rating"].between(0.5, 5.0).all()
print(f"\nAll ratings in [0.5, 5.0]: {in_range}")
print(f"All ratings on 0.5 steps:   {valid_half_steps}")

In [ ]:
ratings["timestamp"] = pd.to_datetime(ratings["timestamp"])
ratings["rating_year"] = ratings["timestamp"].dt.year

print(f"Timestamp range: {ratings['timestamp'].min()} → {ratings['timestamp'].max()}")
print("\nRatings by rating_year (top 10):")
print(ratings["rating_year"].value_counts().sort_index().tail(10))

ratings_per_movie = ratings.groupby("movieId").size()
ratings_per_user = ratings.groupby("userId").size()

print("\nRatings per movie:")
print(ratings_per_movie.describe())
print("\nRatings per user:")
print(ratings_per_user.describe())

## 2. Movies analysis (`movies.csv`)

Columns: `movieId`, `title`, `genres`

In [ ]:
movies.head()

In [ ]:
print("Missing values:")
print(movies.isna().sum())

dup_movie_ids = movies["movieId"].duplicated().sum()
print(f"\nDuplicate movieId values: {dup_movie_ids:,}")
print(f"Total movies: {len(movies):,}")

In [ ]:
TITLE_YEAR_PATTERN = re.compile(r"\((\d{4})\)\s*$")


def extract_release_year(title: str) -> int | None:
    match = TITLE_YEAR_PATTERN.search(str(title))
    return int(match.group(1)) if match else None


def parse_genres(genres: str) -> list[str]:
    if pd.isna(genres) or genres == "(no genres listed)":
        return []
    return [g.strip() for g in str(genres).split("|") if g.strip()]


movies["release_year"] = movies["title"].map(extract_release_year)
movies["genres_list"] = movies["genres"].map(parse_genres)
movies["genre_count"] = movies["genres_list"].map(len)

parsed_years = movies["release_year"].notna().sum()
no_genre = (movies["genres"] == "(no genres listed)").sum()

print(f"Titles with parsed release year: {parsed_years:,} ({parsed_years / len(movies):.1%})")
print(f"Movies with (no genres listed):  {no_genre:,}")
print(f"\nRelease year range: {movies['release_year'].min():.0f} → {movies['release_year'].max():.0f}")
print("\nSample parsed movies:")
movies[["movieId", "title", "release_year", "genres_list"]].head(10)

In [ ]:
genre_counts = (
    movies.explode("genres_list")
    .dropna(subset=["genres_list"])
    .groupby("genres_list")
    .size()
    .sort_values(ascending=False)
)

print("Movies per genre (top 15):")
print(genre_counts.head(15))

## 3. Tags analysis (`tags.csv`)

Columns: `userId`, `movieId`, `tag`, `timestamp`

In [ ]:
tags.head()

In [ ]:
print("Missing values:")
print(tags.isna().sum())

print(f"\nTag records:   {len(tags):,}")
print(f"Unique tags:   {tags['tag'].nunique():,}")
print(f"Unique movies: {tags['movieId'].nunique():,}")
print(f"Unique users:  {tags['userId'].nunique():,}")

duplicate_tag_rows = tags.duplicated().sum()
print(f"\nDuplicate rows (exact): {duplicate_tag_rows:,}")

In [ ]:
tags["tag_normalized"] = tags["tag"].str.strip().str.lower()

raw_unique = tags["tag"].nunique()
normalized_unique = tags["tag_normalized"].nunique()
print(f"Unique tags (raw):        {raw_unique:,}")
print(f"Unique tags (normalized): {normalized_unique:,}")
print(f"Reduction after lowercasing: {raw_unique - normalized_unique:,}")

print("\nMost common tags (normalized, top 20):")
print(tags["tag_normalized"].value_counts().head(20))

In [ ]:
tags["timestamp"] = pd.to_datetime(tags["timestamp"])
tags_per_movie = tags.groupby("movieId").size()

print(f"Tag timestamp range: {tags['timestamp'].min()} → {tags['timestamp'].max()}")
print("\nTags per movie:")
print(tags_per_movie.describe())

## 4. Cross-file checks

Verify joins and orphan references across files.

In [ ]:
movie_ids = set(movies["movieId"])
rated_movie_ids = set(ratings["movieId"].unique())
tagged_movie_ids = set(tags["movieId"].unique())

movies_no_ratings = movie_ids - rated_movie_ids
movies_no_tags = movie_ids - tagged_movie_ids
ratings_unknown_movies = rated_movie_ids - movie_ids
tags_unknown_movies = tagged_movie_ids - movie_ids

print(f"Movies in catalog:              {len(movie_ids):,}")
print(f"Movies with at least one rating: {len(rated_movie_ids):,}")
print(f"Movies with at least one tag:    {len(tagged_movie_ids):,}")
print(f"\nMovies with no ratings:          {len(movies_no_ratings):,}")
print(f"Movies with no tags:             {len(movies_no_tags):,}")
print(f"Ratings referencing unknown movieId: {len(ratings_unknown_movies):,}")
print(f"Tags referencing unknown movieId:    {len(tags_unknown_movies):,}")

In [ ]:
ratings_with_movies = ratings.merge(movies, on="movieId", how="inner")
tags_with_movies = tags.merge(movies, on="movieId", how="inner")

print(f"Ratings joined to movies: {len(ratings_with_movies):,} / {len(ratings):,}")
print(f"Tags joined to movies:    {len(tags_with_movies):,} / {len(tags):,}")

if ratings_unknown_movies:
    print(f"\nSample unknown movieIds in ratings: {sorted(ratings_unknown_movies)[:10]}")
if tags_unknown_movies:
    print(f"Sample unknown movieIds in tags:    {sorted(tags_unknown_movies)[:10]}")

## 5. Data quality summary

Key findings for the team before Stage 2 (schema lock).

In [ ]:
summary = {
    "ratings_rows": len(ratings),
    "ratings_users": ratings["userId"].nunique(),
    "ratings_movies": ratings["movieId"].nunique(),
    "ratings_missing": int(ratings.isna().sum().sum()),
    "ratings_duplicates": int(ratings.duplicated().sum()),
    "rating_min": float(ratings["rating"].min()),
    "rating_max": float(ratings["rating"].max()),
    "rating_year_min": int(ratings["rating_year"].min()),
    "rating_year_max": int(ratings["rating_year"].max()),
    "movies_rows": len(movies),
    "movies_missing": int(movies.isna().sum().sum()),
    "movies_duplicate_ids": int(movies["movieId"].duplicated().sum()),
    "movies_no_genres_listed": int((movies["genres"] == "(no genres listed)").sum()),
    "movies_with_release_year": int(movies["release_year"].notna().sum()),
    "tags_rows": len(tags),
    "tags_unique_raw": int(tags["tag"].nunique()),
    "tags_unique_normalized": int(tags["tag_normalized"].nunique()),
    "movies_no_ratings": len(movies_no_ratings),
    "movies_no_tags": len(movies_no_tags),
    "ratings_orphan_movie_ids": len(ratings_unknown_movies),
    "tags_orphan_movie_ids": len(tags_unknown_movies),
}

summary_df = pd.Series(summary, name="value").to_frame()
summary_df

### Takeaways for Elasticsearch schema (Stage 2)

1. **`rating_year`** — derive from rating `timestamp` (calendar year of when the rating was submitted).
2. **`release_year`** — parse from movie title pattern `(YYYY)` at end of string; not all titles may parse.
3. **`genres`** — pipe-delimited in CSV; parse to keyword array; handle `(no genres listed)`.
4. **`tags`** — free text; normalize to lowercase for aggregation and search.
5. **Joins** — `movieId` is the primary key across all three files; check orphan counts above.
6. **Rating values** — expect 0.5–5.0 in half-star steps; validate in ETL.

See also: `docs/data_quality_summary.md`